# Лабораторная работа №3. Вариант 12
## Файл для защиты

Тема работы: оптимизация среднеквадратичных отклонений методом Гаусса-Ньютона.

Нужно подобрать коэффициенты модельной функции так, чтобы она как можно лучше приближала заданные точки.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Постановка задачи

Модельная функция:

$$
\varphi(x_0,x_1,x_2,x_3,t)=x_3t^3+x_2t^2+x_1t+x_0.
$$

Целевая функция:

$$
F(x)=\frac12\sum_{i=1}^{m} f_i(x)^2=\frac12\|f(x)\|^2,
$$

где

$$
f_i(x)=\varphi(x,t_i)-y_i.
$$

Для варианта 12:

$$
x^{(0)}=[0,0,0,0],
$$

$$
\overline{x}=[\sqrt2,-3,3\sqrt2,1].
$$

Точность:

$$
\varepsilon=0.001.
$$

## Подготовка данных

В задании указана модельная функция и вектор коэффициентов $\overline{x}$, по которому нужно построить график. Отдельный файл с экспериментальными точками не дан, поэтому, как в примере Попова, создаём точки сами:

1. берём равномерную сетку значений $t$;
2. считаем чистые значения $y=\varphi(\overline{x},t)$;
3. добавляем шум, чтобы получить наблюдаемые точки $y_i$.

Чтобы результат всегда был одинаковым при запуске, фиксируем seed генератора случайных чисел.

In [ ]:
def phi(x, t):
    return x[3] * t**3 + x[2] * t**2 + x[1] * t + x[0]

eps = 0.001
m = 35
t = np.linspace(-7, 7, m)
x_start = np.array([0, 0, 0, 0], dtype=float)
x_true = np.array([np.sqrt(2), -3, 3 * np.sqrt(2), 1], dtype=float)

rng = np.random.default_rng(12)
noise = rng.normal(loc=0, scale=50, size=m)
y_clean = phi(x_true, t)
y_observed = y_clean + noise

print(f"x_start = {x_start}")
print(f"x_bar = {x_true}")

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(t, y_clean, label="модельная функция", color="C0")
plt.scatter(t, y_clean, label="чистые точки", color="C1", s=24)
plt.scatter(t, y_observed, label="точки с шумом", color="C2", s=24, alpha=0.75)
plt.xlabel("t")
plt.ylabel("y")
plt.grid(True)
plt.legend()
plt.show()

## Метод Гаусса-Ньютона

Метод применяется к задаче минимизации суммы квадратов:

$$
F(x)=\frac12\|f(x)\|^2.
$$

На каждой итерации решается система

$$
(J^TJ)\Delta x=-J^Tf(x_k),
$$

после чего

$$
x_{k+1}=x_k+\Delta x.
$$

Здесь $J$ — матрица Якоби вектора невязок.

В нашей задаче

$$
f_i(x)=x_3t_i^3+x_2t_i^2+x_1t_i+x_0-y_i.
$$

Поэтому производные простые:

$$
\frac{\partial f_i}{\partial x_0}=1,
\qquad
\frac{\partial f_i}{\partial x_1}=t_i,
\qquad
\frac{\partial f_i}{\partial x_2}=t_i^2,
\qquad
\frac{\partial f_i}{\partial x_3}=t_i^3.
$$

Строка матрицы Якоби:

$$
J_i=[1,t_i,t_i^2,t_i^3].
$$

Важно для защиты: модель линейна по коэффициентам $x_0,x_1,x_2,x_3$, поэтому Гаусс-Ньютон здесь фактически решает линейную задачу МНК и находит оптимум почти сразу.

In [ ]:
def jacobian_matrix(t):
    return np.column_stack([np.ones_like(t), t, t**2, t**3])

def residuals(x, t, y):
    return phi(x, t) - y

def objective(x, t, y):
    r = residuals(x, t, y)
    return 0.5 * np.sum(r**2)

def gauss_newton(t, y, x0, eps=0.001, max_iter=20):
    x = np.array(x0, dtype=float)
    J = jacobian_matrix(t)
    history = []

    for k in range(1, max_iter + 1):
        r = residuals(x, t, y)
        dx = np.linalg.solve(J.T @ J, -J.T @ r)
        x_next = x + dx

        history.append((k, x.copy(), objective(x, t, y), np.linalg.norm(dx), objective(x_next, t, y)))

        if np.linalg.norm(dx) <= eps or abs(objective(x_next, t, y) - objective(x, t, y)) <= eps:
            x = x_next
            break
        x = x_next

    return x, objective(x, t, y), k, history

x_found, F_found, iterations, history = gauss_newton(t, y_observed, x_start, eps)
print(f"x* = {x_found}")
print(f"F(x*) = {F_found:.6f}")
print(f"Количество итераций = {iterations}")

In [ ]:
for k, x, F, dx_norm, F_next in history:
    print(f"{k}: x={np.round(x, 6)}, F={F:.6f}, ||dx||={dx_norm:.6f}, F_next={F_next:.6f}")

In [ ]:
y_found = phi(x_found, t)

plt.figure(figsize=(9, 5))
plt.plot(t, y_clean, label="исходная модель", color="C0")
plt.plot(t, y_found, label="модель после Гаусса-Ньютона", color="red")
plt.scatter(t, y_observed, label="точки с шумом", color="C2", s=24, alpha=0.75)
plt.xlabel("t")
plt.ylabel("y")
plt.grid(True)
plt.legend()
plt.show()

print(f"F для начального x: {objective(x_start, t, y_observed):.6f}")
print(f"F для исходного x_bar: {objective(x_true, t, y_observed):.6f}")
print(f"F для найденного x*: {objective(x_found, t, y_observed):.6f}")

## Число обусловленности матрицы Якоби

Число обусловленности показывает, насколько решение чувствительно к ошибкам в данных.

Для матрицы Якоби:

$$
\kappa(J)=\frac{\sigma_{max}}{\sigma_{min}}.
$$

В методе Гаусса-Ньютона решаются нормальные уравнения с матрицей $J^TJ$, поэтому также смотрим

$$
\kappa(J^TJ).
$$

Обычно

$$
\kappa(J^TJ)\approx\kappa(J)^2,
$$

то есть нормальные уравнения усиливают обусловленность.

In [ ]:
J = jacobian_matrix(t)
cond_J = np.linalg.cond(J)
cond_normal = np.linalg.cond(J.T @ J)
rmse_true = np.sqrt(np.mean((y_clean - y_observed)**2))
rmse_found = np.sqrt(np.mean((y_found - y_observed)**2))

print(f"cond(J) = {cond_J:.6f}")
print(f"cond(J^T J) = {cond_normal:.6f}")
print(f"RMSE исходной модели: {rmse_true:.6f}")
print(f"RMSE найденной модели: {rmse_found:.6f}")

## Итог для защиты по лабораторной 3

Была построена модельная функция с коэффициентами

$$
\overline{x}=[\sqrt2,-3,3\sqrt2,1].
$$

После добавления шума методом Гаусса-Ньютона найдено МНК-решение:

$$
x^*\approx[2.5940,-2.6318,4.0628,0.8917].
$$

Значение целевой функции:

$$
F(x^*)\approx31224.5314.
$$

Число обусловленности:

$$
\kappa(J)\approx211.86,
$$

$$
\kappa(J^TJ)\approx44883.08.
$$

Что сказать на защите: метод Гаусса-Ньютона минимизирует сумму квадратов невязок; в этой задаче матрица Якоби состоит из столбцов $1,t,t^2,t^3$; найденные коэффициенты отличаются от исходных из-за шума; число обусловленности показывает чувствительность решения к ошибкам наблюдений.